# Pascal VOC 2007 Semantic Segmentation (v5)

**Changes from v4:**
- DeepLabV3+: LR 1e-4 → **5e-5** (reduce overfitting, gen_gap was 0.43)
- SAM2: **Large model** + **LR 1e-3** + concat head (128ch x3)
- Loss: **1.0 CE + 1.0 Dice** (equal weights)
- All models: **80 epochs** (consistent for loss curve comparison)
- Image size 512, augmentation

## 0. Setup & Dataset

In [ ]:
import torch, os, sys, pathlib, json, glob, shutil, random
import numpy as np, matplotlib.pyplot as plt

# GPU
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Repo
REPO_DIR = "/content/261-mini2"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/gracee-chen/261-mini2.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)

# Dependencies
!pip install -q segmentation-models-pytorch timm transformers scipy
!pip install -q git+https://github.com/facebookresearch/sam2.git

# Find VOC dataset
VOC_ROOT = None
for candidate in [
    os.path.join(REPO_DIR, "voc_data"),
    os.path.join(REPO_DIR, "voc_data", "VOCtrainval_06-Nov-2007"),
    os.path.join(REPO_DIR, "VOCtrainval_06-Nov-2007"),
]:
    if os.path.isdir(os.path.join(candidate, "VOCdevkit", "VOC2007", "JPEGImages")):
        VOC_ROOT = candidate; break
if VOC_ROOT is None:
    for p in pathlib.Path(REPO_DIR).rglob("VOCdevkit"):
        if (p / "VOC2007" / "JPEGImages").is_dir():
            VOC_ROOT = str(p.parent); break
if VOC_ROOT is None:
    print("Dataset not found. Downloading from Kaggle...")
    !pip install -q kaggle
    from google.colab import files; files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    EXTRACT = os.path.join(REPO_DIR, "voc_data")
    !kaggle datasets download -d zaraks/pascal-voc-2007 -p /tmp/voc && unzip -q /tmp/voc/pascal-voc-2007.zip -d {EXTRACT} && rm -rf /tmp/voc
    for c in [EXTRACT, os.path.join(EXTRACT, "VOCtrainval_06-Nov-2007")]:
        if os.path.isdir(os.path.join(c, "VOCdevkit", "VOC2007", "JPEGImages")):
            VOC_ROOT = c; break

assert VOC_ROOT, "Could not find VOCdevkit/VOC2007/JPEGImages"
print(f"VOC_ROOT: {VOC_ROOT}  |  Images: {len(os.listdir(os.path.join(VOC_ROOT,'VOCdevkit','VOC2007','JPEGImages')))}")

# Paths
CKPT = "checkpoints_v5"
RES  = "results_v5"
IMG_SIZE = 512
EPOCHS = 80
SAM2_CKPT = os.path.join(REPO_DIR, "sam2.1_hiera_large.pt")
SAM2_CFG  = "configs/sam2.1/sam2.1_hiera_l.yaml"
if not os.path.exists(SAM2_CKPT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt -O {SAM2_CKPT}
os.makedirs(RES, exist_ok=True)
print(f"Checkpoints: {CKPT}/  |  Results: {RES}/  |  Image size: {IMG_SIZE}  |  Epochs: {EPOCHS}")

## 1. Dataset Exploration

In [ ]:
sys.path.insert(0, REPO_DIR)
from dataset.voc_dataset import VOC_CLASSES, NUM_CLASSES, get_datasets, mask_to_class_index
from torch.utils.data import DataLoader

train_ds, val_ds = get_datasets(VOC_ROOT, image_size=IMG_SIZE)
print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)} (test set)  |  Classes: {NUM_CLASSES}")

explore_dir = f"{RES}/exploration"; os.makedirs(explore_dir, exist_ok=True)
mean = torch.tensor([.485,.456,.406]).view(3,1,1)
std  = torch.tensor([.229,.224,.225]).view(3,1,1)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for col, idx in enumerate(random.sample(range(len(train_ds)), 4)):
    img, mask = train_ds[idx]
    img_np = (img*std+mean).clamp(0,1).permute(1,2,0).numpy()
    mask_np = mask_to_class_index(mask).numpy(); md = mask_np.copy(); md[md==255]=0
    axes[0,col].imshow(img_np); axes[0,col].set_title(f"#{idx}"); axes[0,col].axis("off")
    axes[1,col].imshow(md, cmap="tab20", vmin=0, vmax=20); axes[1,col].axis("off")
    axes[1,col].set_title(", ".join([VOC_CLASSES[c] for c in np.unique(mask_np) if c<21]))
plt.suptitle("Training Samples: Image + Segmentation Mask", fontsize=14); plt.tight_layout()
plt.savefig(f"{explore_dir}/samples.png", dpi=150, bbox_inches="tight"); plt.show()

counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, masks in DataLoader(train_ds, batch_size=4):
    for m in masks: v = mask_to_class_index(m).numpy(); counts += np.bincount(v[v!=255], minlength=NUM_CLASSES)
pcts = 100*counts/counts.sum()

img_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for i in range(len(train_ds)):
    _, mask = train_ds[i]
    for c in np.unique(mask_to_class_index(mask).numpy()):
        if c < NUM_CLASSES: img_counts[c] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 6))
colors = plt.cm.tab20(np.linspace(0,1,21))
bars1 = ax1.bar(range(NUM_CLASSES), pcts, color=colors)
ax1.set_xticks(range(NUM_CLASSES)); ax1.set_xticklabels(VOC_CLASSES, rotation=45, ha="right", fontsize=8)
ax1.set_ylabel("Pixel %"); ax1.set_title("Pixel-level Class Distribution"); ax1.grid(axis="y", alpha=.3)
for b, p in zip(bars1, pcts):
    if p > 0.3: ax1.text(b.get_x()+b.get_width()/2, b.get_height(), f"{p:.1f}%", ha="center", va="bottom", fontsize=6)
bars2 = ax2.bar(range(NUM_CLASSES), img_counts, color=colors)
ax2.set_xticks(range(NUM_CLASSES)); ax2.set_xticklabels(VOC_CLASSES, rotation=45, ha="right", fontsize=8)
ax2.set_ylabel("# Images"); ax2.set_title(f"Image-level Class Frequency ({len(train_ds)} images)"); ax2.grid(axis="y", alpha=.3)
for b, c in zip(bars2, img_counts):
    if c > 0: ax2.text(b.get_x()+b.get_width()/2, b.get_height(), str(c), ha="center", va="bottom", fontsize=7)
plt.suptitle("Class Distribution (train)", fontsize=13); plt.tight_layout()
plt.savefig(f"{explore_dir}/class_distribution.png", dpi=150, bbox_inches="tight"); plt.show()

print(f"\n{'='*50}")
print(f"  Dataset Summary")
print(f"{'='*50}")
print(f"  Train images     : {len(train_ds)}")
print(f"  Val images       : {len(val_ds)}")
print(f"  Classes          : {NUM_CLASSES} (20 objects + background)")
print(f"{'='*50}")

## 2. Train Models

All models: **80 epochs**, 512x512, augmentation, CE+Dice (1+1)

In [ ]:
# U-Net (ResNet-50) — LR=1e-4
!python train/train_unet.py --voc-root {VOC_ROOT} \
    --epochs {EPOCHS} --batch-size 8 --lr 1e-4 --image-size {IMG_SIZE} \
    --augment --ce-weight 1.0 --dice-weight 1.0 \
    --checkpoint-dir {CKPT}/unet

# DeepLabV3+ (ResNet-50, OS=8) — LR=5e-5 (lower to reduce overfitting)
!python train/train_deeplabv3plus.py --voc-root {VOC_ROOT} \
    --epochs {EPOCHS} --batch-size 8 --lr 5e-5 --image-size {IMG_SIZE} \
    --augment --ce-weight 1.0 --dice-weight 1.0 \
    --checkpoint-dir {CKPT}/deeplabv3plus

In [ ]:
# DINOv2 (ViT-B/14) — frozen backbone, 224x224
!python train/train_dinov2.py --voc-root {VOC_ROOT} \
    --epochs {EPOCHS} --batch-size 16 --lr 1e-3 \
    --augment --ce-weight 1.0 --dice-weight 1.0 \
    --checkpoint-dir {CKPT}/dinov2

In [ ]:
# SAM2-Large — frozen encoder, LR=1e-3
!python train/train_sam2.py --voc-root {VOC_ROOT} \
    --sam2-ckpt {SAM2_CKPT} \
    --sam2-cfg  {SAM2_CFG} \
    --epochs {EPOCHS} --batch-size 4 --lr 1e-3 --image-size {IMG_SIZE} \
    --augment --ce-weight 1.0 --dice-weight 1.0 \
    --checkpoint-dir {CKPT}/sam2

## 3. Evaluation

In [ ]:
available = [(n, f"{CKPT}/{n}/best.pth") for n in ["unet","deeplabv3plus","dinov2","sam2"] if os.path.exists(f"{CKPT}/{n}/best.pth")]
models = [a[0] for a in available]; ckpts = [a[1] for a in available]
print(f"Available: {models}")

mt = " ".join(models); cp = " ".join(ckpts)
sf = f"--sam2-ckpt {SAM2_CKPT} --sam2-cfg {SAM2_CFG}" if "sam2" in models else ""

!python evaluation/metrics/compute_metrics.py \
    --model-type {mt} --checkpoint {cp} \
    --voc-root {VOC_ROOT} --image-size {IMG_SIZE} \
    --output-dir {RES}/metrics {sf}

cd = " ".join([f"{CKPT}/{m}" for m in models])
!python evaluation/compare.py --models {mt} --checkpoint-dirs {cd} --metrics-dir {RES}/metrics --output-dir {RES}/compare

t = f"{RES}/compare/summary_table.txt"
if os.path.exists(t): print(open(t).read())

In [ ]:
for name in models:
    fpath = f"{RES}/metrics/{name}_metrics.json"
    if not os.path.exists(fpath): continue
    m = json.load(open(fpath))
    hd = f"{m['HD95']:.2f}" if not np.isnan(m['HD95']) else 'N/A'
    print(f"\n{'='*55}")
    print(f"  {name.upper()}  |  mIoU={m['mIoU']:.4f}  mDice={m['mDice']:.4f}  PixAcc={m['pixel_accuracy']:.4f}  HD95={hd}")
    print(f"{'='*55}")
    print(f"  {'Class':<15} {'IoU':>8} {'Acc':>8} {'Dice':>8}")
    print(f"  {'-'*42}")
    for i, cls in enumerate(VOC_CLASSES):
        def fmt(v): return f"{v:.4f}" if not np.isnan(v) else "  --  "
        print(f"  {cls:<15} {fmt(m['iou_per_class'][i]):>8} {fmt(m['acc_per_class'][i]):>8} {fmt(m['dice_per_class'][i]):>8}")

In [ ]:
sf_sam2 = f"--sam2-ckpt {SAM2_CKPT} --sam2-cfg {SAM2_CFG}"

for name in models:
    sf2 = sf_sam2 if name == "sam2" else ""
    !python evaluation/metrics/confusion_matrix.py \
        --model-type {name} --checkpoint {CKPT}/{name}/best.pth \
        --voc-root {VOC_ROOT} --image-size {IMG_SIZE} --output-dir {RES}/metrics {sf2}

for name in models:
    sf2 = sf_sam2 if name == "sam2" else ""
    !python inference/infer_{name}.py \
        --checkpoint {CKPT}/{name}/best.pth \
        --voc-root {VOC_ROOT} --num-samples 4 --output-dir {RES}/infer_{name} {sf2}

specs = " ".join([f"{n}:{CKPT}/{n}/best.pth" for n in models])
!python evaluation/visualization/visualize_mosaic.py \
    --voc-root {VOC_ROOT} --models {specs} --num-images 6 \
    --output {RES}/visualization/mosaic.png {sf}

for metric in ["miou", "person"]:
    !python evaluation/visualization/visualize_comparison.py \
        --voc-root {VOC_ROOT} --model-type {mt} --checkpoint {cp} \
        --metric {metric} --output-dir {RES}/visualization {sf}

## 4. Ablation Studies

In [ ]:
!python evaluation/ablation/ablation_backbone.py --voc-root {VOC_ROOT}
!python evaluation/ablation/ablation_loss.py --voc-root {VOC_ROOT}
!python evaluation/ablation/ablation_augmentation.py --voc-root {VOC_ROOT}
!python evaluation/ablation/ablation_pretrain.py --voc-root {VOC_ROOT}
!python evaluation/ablation/ablation_resolution.py --voc-root {VOC_ROOT}

## 5. Display All Plots & Download

In [ ]:
from IPython.display import display, Image as IPImage

all_plots = []
for d in [f"{RES}/exploration", f"{RES}/compare", f"{RES}/metrics", f"{RES}/visualization",
          f"{RES}/infer_unet", f"{RES}/infer_deeplabv3plus", f"{RES}/infer_dinov2", f"{RES}/infer_sam2",
          "results/ablation"]:
    all_plots.extend(sorted(glob.glob(os.path.join(d, "**", "*.png"), recursive=True)))

print(f"Total plots: {len(all_plots)}\n")
for f in all_plots:
    print(f"--- {f} ---"); display(IPImage(filename=f, width=800)); print()

In [ ]:
from google.colab import files
zip_path = "/content/261-mini2-results-v5"
shutil.make_archive(zip_path, "zip", root_dir=REPO_DIR, base_dir=RES)
print(f"Created: {zip_path}.zip")
files.download(f"{zip_path}.zip")